In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import copy
from tqdm import tqdm

In [2]:
# Dataset directory relative to this script
data_dir = './dataset'

# Device config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
image_size = 224
batch_size = 32
num_epochs = 10
learning_rate = 1e-4

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load datasets
train_dataset = datasets.ImageFolder(f'{data_dir}/Train', transform=train_transform)
val_dataset = datasets.ImageFolder(f'{data_dir}/Validation', transform=test_transform)
test_dataset = datasets.ImageFolder(f'{data_dir}/Test', transform=test_transform)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

Using device: cuda


In [3]:
def train_model(model, train_loader, val_loader, epochs=num_epochs, lr=learning_rate):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_iter = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training")
        for imgs, labels in train_iter:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            train_iter.set_postfix(loss=loss.item())

        epoch_loss = running_loss / len(train_loader.dataset)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            val_iter = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - Validation")
            for imgs, labels in val_iter:
                imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(imgs)
                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f} Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, 'best_model.pth')
            print("Best model saved.")

    print(f"Training complete. Best Val Acc: {best_val_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model

def evaluate_model(model, test_loader):
    model = model.to(device)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        test_iter = tqdm(test_loader, desc="Testing")
        for imgs, labels in test_iter:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    print(f"Test Accuracy: {acc:.4f}")
    return acc

In [4]:
effnet_model = timm.create_model('efficientnet_b0', pretrained=True)
effnet_model.classifier = nn.Linear(effnet_model.classifier.in_features, 2)
print("Training EfficientNet...")
effnet_model = train_model(effnet_model, train_loader, val_loader)
torch.save(effnet_model.state_dict(), 'efficientnet_model_final.pth')
evaluate_model(effnet_model, test_loader)

Training EfficientNet...


Epoch 1/10 - Validation: 100%|██████████| 63/63 [00:19<00:00,  3.30it/s]


Epoch [1/10] Loss: 0.1876 Val Acc: 0.9420
Best model saved.


Epoch 2/10 - Validation: 100%|██████████| 63/63 [00:17<00:00,  3.63it/s]


Epoch [2/10] Loss: 0.0511 Val Acc: 0.9499
Best model saved.


...

Epoch 10/10 - Validation: 100%|██████████| 63/63 [00:17<00:00,  3.61it/s]


Epoch [10/10] Loss: 0.0169 Val Acc: 0.9578
Training complete. Best Val Acc: 0.9603


Testing: 100%|██████████| 32/32 [00:08<00:00,  3.82it/s]

Test Accuracy: 0.7966


0.7966269841269841

In [5]:
# DeiT model
deit_model = timm.create_model('deit_small_patch16_224', pretrained=True)
deit_model.head = nn.Linear(deit_model.head.in_features, 2)
print("Training DeiT...")
deit_model = train_model(deit_model, train_loader, val_loader)
torch.save(deit_model.state_dict(), 'deit_model_final.pth')
evaluate_model(deit_model, test_loader)

Training DeiT...


Epoch 1/10 - Validation: 100%|██████████| 63/63 [00:20<00:00,  3.00it/s]


Epoch [1/10] Loss: 0.1897 Val Acc: 0.8829
Best model saved.


Epoch 2/10 - Validation: 100%|██████████| 63/63 [00:21<00:00,  2.96it/s]


Epoch [2/10] Loss: 0.0760 Val Acc: 0.8571


...

Epoch 10/10 - Validation: 100%|██████████| 63/63 [00:20<00:00,  3.07it/s]


Epoch [10/10] Loss: 0.0182 Val Acc: 0.9425
Training complete. Best Val Acc: 0.9474


Testing: 100%|██████████| 32/32 [00:07<00:00,  4.15it/s]

Test Accuracy: 0.8274


0.8273809523809523